In [ ]:
##tensorflow框架的resnet50v2相较于pytorch框架的resnet50性能更加好，收敛更快，更稳定
#使用pytorch框架实现resnet50v2网络，完成kaggle的猫狗大战竞赛
#正确版本
from sklearn.model_selection import train_test_split
import glob
import os.path
import cv2
from PIL import Image
from tqdm import tqdm
from matplotlib import pyplot as plt
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import pandas as pd

# ---------------- 1. 全局超参 ----------------
Data_dir = "./data/dogs-vs-cats/train/train"
img_size = 256
batch_size = 32
seed = 42

# 获取原图片信息
all_image = glob.glob(os.path.join(Data_dir + "/*.jpg"))
img = Image.open(all_image[0])
labels = []

# 数据准备,给图片打标签与图片对应
for p in all_image:
    if "dog" in os.path.basename(p):
        labels.append(1)
    else:
        labels.append(0)

# 划分训练集与测试集比例0.15
train_paths, val_paths, train_labels, val_labels = train_test_split(
    all_image, labels, test_size=0.15, stratify=labels, random_state=seed
)
# 
# 自制数据集
class ImageDataset(Dataset):
    def __init__(self, paths, labels=None, is_train=True, transform=None, target_transform=None):
        self.paths = paths
        self.labels = labels
        self.is_train = is_train
        self.transform = transform
        self.target_transform = target_transform

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        img_path = self.paths[idx]
        # 解码（PIL）+ 预处理
        image = Image.open(img_path).convert('RGB')
        if self.transform is not None:
            image = self.transform(image)
        label = torch.tensor(self.labels[idx], dtype=torch.long)
        return image,label
     # 如果提供了标签则返回，否则返回图像和占位符标签
     #    if self.labels is not None:
     #        label = torch.tensor(self.labels[idx], dtype=torch.long)
     #        return image, label
     #    else:
     #        # 测试时没有标签，返回图像和-1作为占位符
     #        return image, torch.tensor(-1, dtype=torch.long)
# 数据预处理管道
pipline_train = transforms.Compose([
    transforms.RandomHorizontalFlip(p=0.5),  # 随机水平翻转
    transforms.RandomRotation(10),  # 随机旋转
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),  # 颜色增强
    transforms.Resize((224, 224)),  # 调整尺寸
    transforms.ToTensor(),  # 转换为张量
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])  # ImageNet标准化
])

pipline_test = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),
])

train_dataset = ImageDataset(train_paths, train_labels, is_train=True, transform=pipline_train)
val_dataset = ImageDataset(val_paths, val_labels, is_train=False, transform=pipline_test)
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, pin_memory=True)

# 数据集构建并加载完毕，开始构建模型
from torchvision.models import resnet50

class BottleneckBlock(nn.Module):
    """
    ResNet50V2 的瓶颈块
    """
    def __init__(self, in_channels, out_channels, stride=1, expansion=4):
        super(BottleneckBlock, self).__init__()

        # 扩展因子
        self.expansion = expansion
        expanded_channels = out_channels * expansion

        # 主分支
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_channels)

        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, 
                               stride=stride, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_channels)

        self.conv3 = nn.Conv2d(out_channels, expanded_channels, kernel_size=1, bias=False)
        self.bn3 = nn.BatchNorm2d(expanded_channels)

        # 跳跃连接
        self.shortcut = nn.Sequential()
        if stride != 1 or in_channels != expanded_channels:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_channels, expanded_channels, kernel_size=1, 
                          stride=stride, bias=False),
                nn.BatchNorm2d(expanded_channels)
            )

    def forward(self, x):
        residual = x

        # 主分支：1x1 -> 3x3 -> 1x1
        out = F.relu(self.bn1(self.conv1(x)))
        out = F.relu(self.bn2(self.conv2(out)))
        out = self.bn3(self.conv3(out))

        # 跳跃连接
        out += self.shortcut(residual)
        out = F.relu(out)

        return out

# --------------- 构建模型 ---------------
class ResNet50V2(nn.Module):
    def __init__(self, dropout=0.2, wd=5e-4, num_classes=2):
        super().__init__()
        # 1. 使用预训练的ResNet50作为backbone
        self.backbone = resnet50(pretrained=True)
        # 去掉最后的全连接层和平均池化层
        self.backbone = nn.Sequential(*list(self.backbone.children())[:-2])

        # 2. 冻结前90%的层
        total_params = len(list(self.backbone.parameters()))
        freeze_until = int(total_params * 0.9)
        for i, param in enumerate(self.backbone.parameters()):
            if i < freeze_until:
                param.requires_grad = False

        # 3. 自定义头部
        self.pool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc1 = nn.Linear(2048, 32)
        self.relu = nn.ReLU(inplace=True)
        self.drop = nn.Dropout(dropout)
        self.fc2 = nn.Linear(32, num_classes)

    def forward(self, x):
        x = self.backbone(x)     # [B, 2048, 7, 7]
        x = self.pool(x)         # [B, 2048, 1, 1]
        x = x.view(x.size(0), -1)
        x = self.fc1(x)
        x = self.relu(x)
        x = self.drop(x)
        x = self.fc2(x)
        # 移除sigmoid，因为CrossEntropyLoss会自动处理
        return x

# # 创建模型
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = ResNet50V2(num_classes=2)  # 二分类
model.to(device)
loss_func = nn.CrossEntropyLoss()    # 自动处理 softmax
# print(f"自定义 ResNet50V2 参数数量: {sum(p.numel() for p in model.parameters()):,}")
optimizer = torch.optim.Adam(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=1e-6, weight_decay=5e-4
)

# 注意：summary需要正确的输入尺寸
try:
    from torchsummary import summary
    summary(model, input_size=(3, 224, 224))
except:
    print("无法显示模型摘要，继续训练...")

class EarlyStopping:
    def __init__(self, patience=1, delta=0, path='checkpoint.pt'):
        """
        patience: 容忍多少个 epoch 没有提升
        delta   : 提升阈值（绝对值）
        path    : 保存最优权重的路径
        """
        self.patience = patience
        self.delta = delta
        self.counter = 0
        self.best_score = None
        self.early_stop = False
        self.val_loss_min = np.inf
        self.path = path

    def __call__(self, val_loss, model):
        score = -val_loss

        if self.best_score is None:
            self.best_score = score
            self.save_checkpoint(val_loss, model)
        elif score < self.best_score + self.delta:
            self.counter += 1
            if self.counter >= self.patience:
                self.early_stop = True
        else:
            self.best_score = score
            self.save_checkpoint(val_loss, model)
            self.counter = 0

    def save_checkpoint(self, val_loss, model):
        """保存最优权重"""
        torch.save(model.state_dict(), self.path)
        self.val_loss_min = val_loss

    def load_best_weights(self, model):
        """训练结束后恢复最优权重"""
        model.load_state_dict(torch.load(self.path))

# 模型训练
early_stopping = EarlyStopping(patience=1, path='best.pt')
history = {'train_loss': [], 'train_acc': [], 'val_loss': []}

for epoch in tqdm(range(30)):
    # ---------- 训练 ----------
    model.train()
    train_loss = 0
    total = 0
    correct = 0

    for x, y in train_loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        pred = model(x)
        loss = loss_func(pred, y)
        loss.backward()
        optimizer.step()

        # 计算准确率
        predict = pred.argmax(dim=1)
        total += y.size(0)  # 修复：使用y而不是labels
        correct += (predict == y).sum().item()  # 修复：使用y而不是labels
        train_loss += loss.item()

    train_acc = correct / total
    avg_train_loss = train_loss / len(train_loader)

    # ---------- 验证 ----------
    model.eval()
    val_loss = 0
    with torch.no_grad():
        for x, y in val_loader:
            x, y = x.to(device), y.to(device)
            pred = model(x)
            val_loss += loss_func(pred, y).item()

    val_loss /= len(val_loader)

    # 记录历史
    history['train_loss'].append(avg_train_loss)
    history['train_acc'].append(train_acc)
    history['val_loss'].append(val_loss)

    print(f'Epoch {epoch+1:02d} | train_loss={avg_train_loss:.4f} | train_acc={train_acc:.4f} | val_loss={val_loss:.4f}')

    # ---------- EarlyStopping ----------
    early_stopping(val_loss, model)
    if early_stopping.early_stop:
        print("Early stopping")
        break

# 训练结束后恢复最优权重
early_stopping.load_best_weights(model)
# 保存模型
torch.save(model, 'resnet50v2-pytorch.pth')

# 绘制训练历史
plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.plot(history['train_acc'], label="Train Acc")
plt.title("Accuracy")
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history['train_loss'], label="Train Loss")
plt.plot(history['val_loss'], label="Val Loss")
plt.title("Loss")
plt.legend()

plt.show()

# 开始测试    
# 测试数据
# model = torch.load("resnet50v2-pytorch.pth",weights_only=False, map_location=device)
test_dir = "./data/dogs-vs-cats/test1/test1"
test_paths = sorted(glob.glob(os.path.join(test_dir, "*.jpg")), key=lambda x: int(os.path.basename(x).split('.')[0]))
def test_dataset(model, device, test_path):
    model = model.eval().to(device)
    results=[]
    for img_path in test_path:
        image = Image.open(img_path).convert("RGB")
        
        image = pipline_test(image).unsqueeze(0).to(device)  # 添加unsqueeze(0)以匹配模型输入维度
        with torch.no_grad():
            
            output = model(image)
            # test_loss += loss_func(output, image).item()
            predict = output.argmax(dim=1)
            idx = int(os.path.basename(img_path).split('.')[0])
            results.append({
                'id': int(idx),
                'label': int(predict),
            })
    df = pd.DataFrame(results)
    submit_df = df[['id','label']].copy()
    submit_df.to_csv("submit_cat&dog.csv", index=False)
    print(f"预测结果已保存到: submit_cat&dog.csv")
    print("\n预测统计:")
    print(f"总图像数: {len(results)}")
test_dataset(model, device, test_paths)